# RQ3 Complete Analysis

Provides the complete RQ3 conformal calibration and coverage analysis.

**Provenance.** This is an English-facing execution copy of the original project notebook. The computational cells are preserved so that the analysis remains reproducible. Chinese narrative cells have been replaced by concise English stage notes; the original notebook remains unchanged in the source project.

## Analysis stage 1

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "RQ3").exists() and (p / "data" / "processed").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("\u627e\u4e0d\u5230\u9879\u76ee\u6839\u76ee\u5f55。\u8bf7\u4ece data3001-project \u6216\u5176\u5b50\u76ee\u5f55\u8fd0\u884c\u672c Notebook。")

RQ3 = PROJECT_ROOT / "RQ3"
OUTPUT = RQ3 / "output"
OUTPUT.mkdir(exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "PingFang SC"]
plt.rcParams["axes.unicode_minus"] = False

print("\u9879\u76ee\u76ee\u5f55：", PROJECT_ROOT)
print("RQ3 \u76ee\u5f55：", RQ3)
print("\u7ed3\u679c\u76ee\u5f55：", OUTPUT)

## Analysis stage 2

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
inventory = []
for path in sorted(RQ3.iterdir()):
    if path.name.startswith("."):
        continue
    if path.is_file():
        inventory.append({"name": path.name, "type": "file", "size_kB": round(path.stat().st_size / 1024, 1)})
    elif path.is_dir():
        files = list(path.rglob("*"))
        inventory.append({"name": path.name + "/", "type": "directory", "n_files": sum(p.is_file() for p in files)})
display(pd.DataFrame(inventory).fillna(""))

## Analysis stage 3

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
required = {
    "forecast cases": PROJECT_ROOT / "data/processed/forecast_origins.parquet",
    "observed trajectories": PROJECT_ROOT / "data/processed/region_A_observations.parquet",
    "fixed FNN weights": PROJECT_ROOT / "output/rq1/en/fnn_model_parameters.npz",
    "saved RQ1 test errors": PROJECT_ROOT / "output/rq1/en/fnn_test_case_metrics.csv",
    "RQ2 regime labels": PROJECT_ROOT / "RQ2/data/forecast_case_regimes.csv",
}
required_table = pd.DataFrame([
    {"input": name, "available": path.exists(), "path": str(path.relative_to(PROJECT_ROOT))}
    for name, path in required.items()
])
display(required_table)
assert required_table["available"].all(), "\u5b58\u5728\u7f3a\u5931\u8f93\u5165，\u8bf7\u5148\u5b8c\u6210 RQ1/RQ2 \u6570\u636e\u51c6\u5907。"

## Analysis stage 4

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
RUN_PIPELINE = True

if RUN_PIPELINE:
    completed = subprocess.run(
        [sys.executable, str(RQ3 / "run_all.py")],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
        check=True,
    )
    log_lines = completed.stdout.splitlines()
    print("Pipeline \u5b8c\u6210。\u6700\u540e 30 \u884c\u65e5\u5fd7：")
    print("\n".join(log_lines[-30:]))
else:
    print("\u8df3\u8fc7\u91cd\u65b0\u8fd0\u884c，\u8bfb\u53d6 RQ3/output/ \u4e2d\u7684\u73b0\u6709\u7ed3\u679c。")

## Analysis stage 5

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
origins = pd.read_parquet(required["forecast cases"])
split_summary = (
    origins.groupby("partition")
    .agg(usable_drifters=("ID", "nunique"), forecast_cases=("forecast_case_id", "size"))
    .reindex(["training", "calibration", "test"])
)
split_summary.loc["Total"] = split_summary.sum()
display(split_summary)

partition_ids = {part: set(g["ID"]) for part, g in origins.groupby("partition")}
overlap = {
    "training ∩ calibration": len(partition_ids["training"] & partition_ids["calibration"]),
    "training ∩ test": len(partition_ids["training"] & partition_ids["test"]),
    "calibration ∩ test": len(partition_ids["calibration"] & partition_ids["test"]),
}
display(pd.Series(overlap, name="overlapping drifters").to_frame())
assert max(overlap.values()) == 0

## Analysis stage 6

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
quantiles = pd.read_csv(OUTPUT / "rq3_conformal_quantiles.csv")
coverage_lead = pd.read_csv(OUTPUT / "rq3_coverage_by_lead.csv")

q_key = (
    quantiles[quantiles["lead_time_hours"].isin([24, 48, 72])]
    .pivot(index="lead_time_hours", columns="nominal_coverage", values="radius_km")
)
display(q_key.round(2))

main_key = coverage_lead[
    coverage_lead["alpha"].eq(0.10)
    & coverage_lead["lead_time_hours"].isin([24, 48, 72])
][["lead_time_hours", "empirical_coverage", "radius_km", "area_km2", "n_test_cases"]]
display(main_key.round({"empirical_coverage": 3, "radius_km": 2, "area_km2": 0}))

In [ ]:
display(Image(filename=str(OUTPUT / "rq3_coverage_and_radius_by_lead.png"), width=1000))

## Analysis stage 7

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

## Analysis stage 8

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
regime_global = pd.read_csv(OUTPUT / "rq3_coverage_by_regime.csv")
display(regime_global.round(3))
display(Image(filename=str(OUTPUT / "rq3_72h_coverage_by_regime.png"), width=850))

## Analysis stage 9

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

## Analysis stage 10

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
comparison = pd.read_csv(OUTPUT / "rq3_72h_regime_comparison.csv")
overall = pd.read_csv(OUTPUT / "rq3_72h_overall_comparison.csv")
drifter_max = pd.read_csv(OUTPUT / "rq3_drifter_max_coverage.csv")

display(comparison.round(3))
display(overall.round(3))
display(Image(filename=str(OUTPUT / "rq3_regime_coverage_comparison.png"), width=950))

## Analysis stage 11

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
display(drifter_max.round(3))

## Analysis stage 12

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
bootstrap = pd.read_csv(OUTPUT / "rq3_paired_cluster_bootstrap_summary.csv")
focus_metrics = [
    "overall_coverage_diff_pp",
    "overall_mean_area_diff_km2",
    "overall_mean_area_diff_pct",
    "regime_coverage_range_diff_pp",
    "Kuroshio Core__coverage_diff_pp",
    "Shelf__coverage_diff_pp",
    "Transition__coverage_diff_pp",
    "Other Offshore__coverage_diff_pp",
]
focus = bootstrap[bootstrap["metric"].isin(focus_metrics)].set_index("metric").loc[focus_metrics]
display(focus[["estimate", "ci95_lower", "ci95_upper", "valid_bootstrap_reps"]].round(3))

## Analysis stage 13

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

## Analysis stage 14

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
trajectory = pd.read_csv(OUTPUT / "rq3_simultaneous_trajectory_summary.csv")
display(trajectory.round(3))

## Analysis stage 15

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
display(Image(filename=str(OUTPUT / "rq3_representative_case_regions.png"), width=750))

## Analysis stage 16

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
audit = pd.read_csv(OUTPUT / "rq3_integrity_audit_checks.csv")
display(audit)
print(f"Audit \u901a\u8fc7：{int(audit['passed'].sum())}/{len(audit)}")
assert audit["passed"].all(), "\u5b8c\u6574\u6027\u5ba1\u8ba1\u672a\u5168\u90e8\u901a\u8fc7，\u8bf7\u68c0\u67e5 FAIL \u9879。"

In [ ]:
audit_quantiles = pd.read_csv(OUTPUT / "rq3_audit_calibration_quantiles.csv")
audit_case = pd.read_csv(OUTPUT / "rq3_audit_case_level_coverage.csv")
audit_drifter = pd.read_csv(OUTPUT / "rq3_audit_drifter_level_coverage.csv")

display(audit_quantiles.round(3))
display(audit_drifter[[
    "regime", "method", "n_test_drifters", "all_cases_covered_drifters",
    "drifter_level_all_case_coverage",
]].round(3))

## Analysis stage 17

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

## Analysis stage 18

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

## Analysis stage 19

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

## Analysis stage 20

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
output_index = pd.DataFrame([
    {
        "file": p.name,
        "type": p.suffix.lstrip(".").upper(),
        "size_kB": round(p.stat().st_size / 1024, 1),
    }
    for p in sorted(OUTPUT.iterdir()) if p.is_file()
])
display(output_index)
print(f"\u5171 {len(output_index)} \u4e2a\u6b63\u5f0f\u8f93\u51fa\u6587\u4ef6。")